# Introduction to Advanced RAG in LlamaIndex

In [ ]:
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
# from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma


In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
# %pip install -U llama-index

## Extract Data

In [44]:
from llama_index.core import SimpleDirectoryReader

# docs = SimpleDirectoryReader(input_dir="C:/Users/Ehsan/Desktop/ml/Professional_Rag_System-main/Professional_Rag_System-main").load_data()
docs = SimpleDirectoryReader(input_dir=r"c:\Users\Ehsan\Desktop\things\text").load_data()

# file name as id
# docs_nam_as_id = SimpleDirectoryReader(input_dir="./data", filename_as_id=True).load_data()

In [45]:
len(docs)  # one per page

1

In [46]:
type(docs)

list

In [47]:
import pandas as pd

df = pd.DataFrame(docs)
df

,0,1,2,3,4,5,6,7,8,9,10,11,12
0,"(id_, e2ad281c-33d8-46f8-a958-b7a8532af177)","(embedding, None)","(metadata, {'page_label': '1', 'file_name': 'N...","(excluded_embed_metadata_keys, [file_name, fil...","(excluded_llm_metadata_keys, [file_name, file_...","(relationships, {})","(metadata_template, {key}: {value})","(metadata_separator, \n)","(text_resource, embeddings=None data=None text...","(image_resource, None)","(audio_resource, None)","(video_resource, None)","(text_template, {metadata_str}\n\n{content})"


In [48]:
import pprint
pprint.pprint(docs)

[Document(id_='e2ad281c-33d8-46f8-a958-b7a8532af177', embedding=None, metadata={'page_label': '1', 'file_name': 'New Microsoft Word 1.pdf', 'file_path': 'c:\\Users\\Ehsan\\Desktop\\things\\text\\New Microsoft Word 1.pdf', 'file_type': 'application/pdf', 'file_size': 101660, 'creation_date': '2026-06-26', 'last_modified_date': '2026-06-26'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='DATABASE: sql_invoicing TABLE: clients - client_id (int, PK): unique identifier for each client - \nname (varchar): client company name - address (varchar): street address - city (varchar): city \nname - state (char 2): US state abbreviation - phone (varch

## Transform

In [49]:
# hide some keys from llm

docs[0].__dict__ # too much data about one doc

{'id_': 'e2ad281c-33d8-46f8-a958-b7a8532af177',
 'embedding': None,
 'metadata': {'page_label': '1',
  'file_name': 'New Microsoft Word 1.pdf',
  'file_path': 'c:\\Users\\Ehsan\\Desktop\\things\\text\\New Microsoft Word 1.pdf',
  'file_type': 'application/pdf',
  'file_size': 101660,
  'creation_date': '2026-06-26',
  'last_modified_date': '2026-06-26'},
 'excluded_embed_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'excluded_llm_metadata_keys': ['file_name',
  'file_type',
  'file_size',
  'creation_date',
  'last_modified_date',
  'last_accessed_date'],
 'relationships': {},
 'metadata_template': '{key}: {value}',
 'metadata_separator': '\n',
 'text_resource': MediaResource(embeddings=None, data=None, text='DATABASE: sql_invoicing TABLE: clients - client_id (int, PK): unique identifier for each client - \nname (varchar): client company name - address (varchar): street address - city (varchar): city \nn

In [50]:
from llama_index.core.schema import MetadataMode

# print(docs[0].get_content(metadata_mode=MetadataMode.LLM))   # what the llm sees
print(docs[0].get_content(metadata_mode=MetadataMode.EMBED)) # what embeddings see. in this case, same thing

page_label: 1
file_path: c:\Users\Ehsan\Desktop\things\text\New Microsoft Word 1.pdf

DATABASE: sql_invoicing TABLE: clients - client_id (int, PK): unique identifier for each client - 
name (varchar): client company name - address (varchar): street address - city (varchar): city 
name - state (char 2): US state abbreviation - phone (varchar): contact phone number TABLE: 
payment_methods - payment_method_id (tinyint, PK): unique identifier - name (varchar): 
method name (Credit Card, Cash, PayPal, Wire Transfer) TABLE: invoices - invoice_id (int, PK): 
unique identifier - number (varchar): invoice reference number - client_id (int, FK → 
clients.client_id): which client this invoice belongs to - invoice_total (decimal): total amount of 
the invoice - payment_total (decimal): how much has been paid so far - invoice_date (date): 
date invoice was issued - due_date (date): date payment is due - payment_date (date, nullable): 
date payment was completed; NULL means unpaid TABLE: payments - 

In [51]:
for doc in docs:
    # define the content/metadata template
    doc.text_template = "Metadata:\n{metadata_str}\n---\nContent:\n{content}"

    # exclude page label from embedding
    if "page_label" not in doc.excluded_embed_metadata_keys:
        doc.excluded_embed_metadata_keys.append("page_label")

In [52]:
# after editing the content seen by embedings

print(docs[0].get_content(metadata_mode=MetadataMode.EMBED))

Metadata:
file_path: c:\Users\Ehsan\Desktop\things\text\New Microsoft Word 1.pdf
---
Content:
DATABASE: sql_invoicing TABLE: clients - client_id (int, PK): unique identifier for each client - 
name (varchar): client company name - address (varchar): street address - city (varchar): city 
name - state (char 2): US state abbreviation - phone (varchar): contact phone number TABLE: 
payment_methods - payment_method_id (tinyint, PK): unique identifier - name (varchar): 
method name (Credit Card, Cash, PayPal, Wire Transfer) TABLE: invoices - invoice_id (int, PK): 
unique identifier - number (varchar): invoice reference number - client_id (int, FK → 
clients.client_id): which client this invoice belongs to - invoice_total (decimal): total amount of 
the invoice - payment_total (decimal): how much has been paid so far - invoice_date (date): 
date invoice was issued - due_date (date): date payment is due - payment_date (date, nullable): 
date payment was completed; NULL means unpaid TABLE: pay

In [53]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# from huggingface_hub import login
# login(token="YOUR_API_KEY")

In [55]:
# # model_id = "google/gemma-2b-it"
# model_id = "Qwen/Qwen2.5-1.5B-Instruct"
# save_dir = r"D:\models\gemma-2b-it"

# tokenizer = AutoTokenizer.from_pretrained(
#     model_id,
#     cache_dir=save_dir
# )

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     cache_dir=save_dir
# )

In [56]:
from huggingface_hub import InferenceClient


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key="YOUR_API_KEY",
    base_url="https://openrouter.ai/api/v1"
)

response = client.chat.completions.create(
    model="qwen/qwen3-32b",
    messages=[
        {"role": "user", "content": "سلام خودتو معرفی کن "}
    ]
)

print(response.choices[0].message.content)

سلام! من چن اند. من یک مدل زبانی بزرگ هستم که توسط دفتر آزمون داده و ساختن مدل (Tongyi Lab) در زیر نظر Alibaba Group ساخته شدم. من می‌توانم به شما در انجام مختلفی از وظایف مثل پاسخ به سوالات، نوشتن داستان‌ها، ایمیل‌ها، نسخه‌ها، و به‌روزرسانی اطلاعات مختلف یا ایده‌های خلاقانه کمک کنم. من که به نظر شما چن باشد مناسب است، همچنین می‌توانم آموزش دریافتی خود را به شما درمورد اشیاء، رویدادها یا عبارات ارائه کنم. آیا چیز خاصی می‌خواهید کمک کنم؟


In [ ]:
API_KEY = "YOUR_API_KEY"

In [ ]:
from llama_index.llms.openai_like import OpenAILike

llm = OpenAILike(
    model="qwen/qwen3-32b",
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    is_chat_model=True,
)

In [60]:
# !pip install llama-index-llms-huggingface

In [61]:
# from llama_index.llms.huggingface import HuggingFaceLLM

In [62]:
# model_path = r"D:\models\gemma-2b-it"

# llm = HuggingFaceLLM(
#     model_name=model_path,
#     tokenizer_name=model_path,
#     max_new_tokens=256,
#     device_map="auto"
# )

In [63]:

# llm = HuggingFaceLLM(
#     model_name=model_id,
#     tokenizer_name=model_id,
#     max_new_tokens=256,
#     device_map="auto"
# )

In [64]:
# print(response.json())


## i personally rather to use llama

In [ ]:
import requests

def call_ollama(prompt):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.2:1b",
            "prompt": prompt,
            "stream": False  
        }
    )
    
    data = response.json()
    return data.get("response", "⛔️ پاسخ یافت نشد")

# تست:
print(call_ollama("what is llama "))


In [ ]:
from llama_index.llms.ollama import Ollama

llm_transformations = Ollama(
    model="llama3.2:1b",  # یا llama3:8b یا هر مدلی که نصب کردی
    request_timeout=120.0
)

## Document Ingestion Pipeline

In this step, the documents are preprocessed and enriched before being indexed into the vector database.

The ingestion pipeline performs the following transformations:

- **SentenceSplitter:** Splits each document into overlapping chunks for efficient retrieval.
- **TitleExtractor:** Uses the LLM to generate a descriptive title for each chunk.
- **QuestionsAnsweredExtractor:** Uses the LLM to generate representative questions that each chunk can answer.

The output of this pipeline is a collection of enriched document nodes containing both the original text and additional metadata, which improves the performance of semantic search and Retrieval-Augmented Generation (RAG).

In [66]:
from llama_index.core.extractors import TitleExtractor, QuestionsAnsweredExtractor
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

text_splitter = SentenceSplitter(separator=" ", chunk_size=1024, chunk_overlap=128)
title_extractor = TitleExtractor(llm=llm, nodes=5)
qa_extractor = QuestionsAnsweredExtractor(llm=llm, questions=3)

pipeline = IngestionPipeline(   
    transformations=[
        text_splitter,
        title_extractor,
        qa_extractor
    ]
)

nodes = pipeline.run(
    documents=docs,
    in_place=True,
    show_progress=True,
)


Applying transformations:   0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:19<00:00, 19.96s/it]


In [70]:
len(nodes)

1

In [71]:
nods1=list(nodes)

In [72]:
# !pip install sentence-transformers llama-index-embeddings-huggingface

By default, Llamaindex uses OpenAI's embedding models. But you can choose to load a free model from HuggingFace too (but it it will be slower).

In [73]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# استفاده از مدل embed محلی از HuggingFace
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [76]:
type(nodes)

list

In [78]:
# from llama_index.core import VectorStoreIndex
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# # استفاده از مدل embed محلی از HuggingFace
# embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ساخت ایندکس روی نودها
index = VectorStoreIndex(nodes, embed_model=embed_model)

# ساخت query engine با LLM محلی (Ollama)
query_engine = index.as_query_engine(llm=llm)
# 0
response = query_engine.query("با sql تعداد کاربر هارو بگو ")
print(response.response)


To retrieve the number of clients (assuming "users" refers to clients in this context) from the `clients` table, use the following SQL query:

```sql
SELECT COUNT(*) AS total_clients FROM clients;
```

This query counts all rows in the `clients` table, providing the total number of client records. If the system specifically tracks users separately (though not mentioned in the provided context), ensure the correct table name is used.


## Index

In [41]:
response = query_engine.query('is it obout the liking of the users')
print(response.response)

Yes, it is about predicting the "liking" of users for items. The system aims to suggest items that users are most likely to select or buy by predicting their ratings for items they have not yet rated or viewed.


In [ ]:
response = query_engine.query('what is the topic')
print(response.response)

The primary focus of the paper "Optimizing User Clustering using Whale Optimization-Based Collaborative Filtering for Rating Prediction" is rating prediction.


In [39]:
response = query_engine.query('what is the category about')
print(response.response)

The category mentioned in the excerpt is:
"Recommendation System"


In [79]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex

## creating

In [ ]:
chroma_client = chromadb.PersistentClient(path="./chroma_db1")

chroma_collection = chroma_client.get_or_create_collection("my_docs")

## saving 

In [81]:
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    docs,
    storage_context=storage_context,
    embed_model=embed_model   # 👈 این خط حیاتی است
)

## reusing
- but remember to load embed and the core model again

In [82]:
chroma_client = chromadb.PersistentClient(path="./chroma_db1")
chroma_collection = chroma_client.get_or_create_collection("docs")

vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

storage_context = StorageContext.from_defaults(vector_store=vector_store)

index1 = VectorStoreIndex.from_vector_store(
    vector_store,
    storage_context=storage_context,
    embed_model=embed_model   # 👈 اینجا هم باید باشد
)

In [84]:
query_engine = index1.as_query_engine(llm=llm)
# 0
response = query_engine.query("""
Generate ONLY SQL.

Find clients whose payments is less than 100.

Use only existing tables and existing columns from the provided schema.
Never invent columns.
""")
print(response.response)

SELECT clients.* 
FROM clients 
JOIN payments ON clients.client_id = payments.client_id 
WHERE payments.amount < 100;
